# Small Portfolio Hedging: Neural Networks and Heston Benchmarks

### Overview:
We attempt to hedge a portfolio of 3 call options in a simulated Heston market with transaction costs. We compare paper-style networks trained on static stock paths generated prior to training with our networks trained on stock paths fresh batches generated in-line during training. We evaluate the CVaR and entropic risks for each model, and compare the trained strategies with classical Heston hedges. 

We also compare the time and memory complexity for training both the paper's and our model in order to describe use cases for either model.

# Comparison of Paper's and Our Network's Hedging Strategy for a Small Portfolio

We hedge three European call options over 30 trading days in a simulated Heston market, where both stock price and variance change over time. The underlyings are assumed to be independent. 

The paper-style networks train from a fixed pool of paths. Our networks receive fresh simulated batches at each iteration. Validation selects the final policies, and 8,192 separate test paths measure their performance.

## Imports 

In [ ]:
from scripts.general_nn import generalNN
from scripts import plots
import importlib

from dataclasses import replace

from scripts.general_nn import generalNN
from scripts.settings import NetworkSettings, Settings, StockSettings, InstrumentSettings
from scripts import plots

## Define Synthetic Stock Parameters

In [ ]:
stocks_list = [
    StockSettings(name="Stock1", initial_price=100.0, strike=100.0,
                  initial_variance=0.04, long_run_variance=0.04,
                  mean_reversion=1.0, vol_of_vol=2.0, correlation=-0.7),
    StockSettings(name="Stock2", initial_price=50.0, strike=50.0,
                  initial_variance=0.02, long_run_variance=0.02,
                  mean_reversion=1.2, vol_of_vol=1.8, correlation=-0.6),
    StockSettings(name="Stock3", initial_price=200.0, strike=200.0,
                  initial_variance=0.06, long_run_variance=0.06,
                  mean_reversion=0.8, vol_of_vol=2.2, correlation=-0.5),
]

## Model Setup

### General Settings

In [ ]:
# Use a small training budget for this first experiment.
settings = Settings(
    warmup_updates=100,
    continuation_updates=1000,
    number_of_replicates=1,

    # Check validation performance every 25 training updates.
    validation_interval=50,

    # Set the sizes of the calibration, validation, and test datasets.
    normalization_size=2048,
    validation_size=2048,
    test_size=8192,

    # Used only if we choose to train from a fixed dataset.
    static_training_size=2**18,
)

### Network Settings

In [80]:
n_stocks = len(stocks_list)

network = NetworkSettings(
    hidden_sizes=(12 * n_stocks, 12 * n_stocks),
    activation="tanh",
    batch_normalization=True
)

### Model Initialization

In [81]:
# dynamic_data:
# True: generate fresh training batches during each iteration.
# False: sample batches from one fixed training dataset.

settings = replace(settings, stocks=tuple(stocks_list), network=network)

# Create a new paper-style model that trades using stock and the variance information.
paper = generalNN(
    method="paper",
    settings=settings,
    dynamic_data=False,
    name="Paper network: stock + variance",
)

# Create our new model using the same settings, including the same instruments from the paper example.
ours_settings = replace(
    settings,
    instruments=paper.settings.instruments
)
ours = generalNN(
    method="ours",
    settings=ours_settings,
    dynamic_data=True,
    name="Our network: stock + variance",
)

# Training and Evaluation

In [82]:
# Train and evaluate each newly created model.
for model in (paper, ours):

    print(f"\nTraining: {model.name}")

    # Prepare the training data and fit the hedging policies.
    model.train(verbose=True)

    # Measure the trained policies on separate test paths.
    model.evaluate()


Training: Paper network: stock + variance
paper_full_0_warmup: update 0, validation risk 17.91601
paper_full_0_warmup: update 100, validation risk 10.38330
paper_full_0_CVaR: update 0, validation risk 10.39639
paper_full_0_CVaR: update 500, validation risk 9.00407
paper_full_0_CVaR: update 1000, validation risk 8.72391
paper_full_0_Entropic: update 0, validation risk 28.22055
paper_full_0_Entropic: update 500, validation risk 19.84315
paper_full_0_Entropic: update 1000, validation risk 16.47914

Training: Our network: stock + variance
ours_0_warmup: update 0, validation risk 17.95961
ours_0_warmup: update 100, validation risk 10.52861
ours_0_CVaR: update 0, validation risk 10.53148
ours_0_CVaR: update 500, validation risk 9.07089
ours_0_CVaR: update 1000, validation risk 8.75196
ours_0_Entropic: update 0, validation risk 22.35587
ours_0_Entropic: update 500, validation risk 18.98299
ours_0_Entropic: update 1000, validation risk 16.15580


# Display Experiment Results

In [ ]:
experiments = [paper, ours]

plots.plot_results_grid(experiments)